# Feature Engineering: SaaS Sales Conversations

This notebook performs clear and focused feature engineering on the `DeepMostInnovations/saas-sales-conversations` dataset. The goal is to create meaningful, explainable features combining domain-specific attributes, statistical properties, and semantic text representations, while avoiding over-engineering.

In [1]:
# 1. Load and prepare data
import pandas as pd
import numpy as np
from datasets import load_dataset
import warnings
warnings.filterwarnings('ignore')

# Load dataset
dataset = load_dataset("DeepMostInnovations/saas-sales-conversations", split="train")

# Convert to pandas DataFrame for feature engineering
df = dataset.to_pandas().sample(n=8000, random_state=42)

# Drop pre-existing embeddings and unnecessary JSON columns to start fresh
cols_to_drop = [col for col in df.columns if col.startswith('embedding_')] + ['scenario', 'conversation', 'probability_trajectory', 'conversation_id', 'company_id', 'company_name']
df = df.drop(columns=cols_to_drop)

print(f"Initial shape: {df.shape}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

cleaned_custom_dataset.csv:   0%|          | 0.00/7.17G [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Initial shape: (8000, 10)


In [2]:
# 2. Basic feature preparation
from sklearn.preprocessing import LabelEncoder

# The target 'outcome' is already 0/1. We'll encode small categorical features.
categorical_cols = ['product_name', 'product_type', 'conversation_style', 'conversation_flow', 'communication_channel']

# Using One-Hot Encoding for small categories
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

In [3]:
# 3. Text Feature Engineering (CORE PART)
from sentence_transformers import SentenceTransformer

# Basic statistical text features
df['text_length'] = df['full_text'].astype(str).apply(len)
df['word_count'] = df['full_text'].astype(str).apply(lambda x: len(x.split()))
df['average_word_length'] = df['text_length'] / (df['word_count'] + 1) # Add 1 to avoid division by zero

# Generate text embeddings using a lightweight pre-trained model
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(df['full_text'].tolist(), show_progress_bar=True)

# Store embeddings temporarily as a list of vectors in the dataframe
df['text_embedding'] = list(embeddings)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/250 [00:00<?, ?it/s]

In [4]:
# 4. Interaction Features

# Meaningful combinations based on domain logic
df['engagement_x_length'] = df['customer_engagement'] * df['conversation_length']
df['engagement_per_turn'] = df['customer_engagement'] / (df['conversation_length'] + 1)
df['text_per_turn'] = df['text_length'] / (df['conversation_length'] + 1)

In [5]:
# 5. Distribution-based Transformations
from sklearn.preprocessing import StandardScaler

# Log transform skewed features
skewed_features = ['text_length', 'word_count', 'conversation_length']

for feat in skewed_features:
    df[f'{feat}_log'] = np.log1p(df[feat])

# Standardize continuous features
features_to_scale = ['customer_engagement', 'sales_effectiveness', 'average_word_length',
                     'engagement_x_length', 'engagement_per_turn', 'text_per_turn'] + [f'{f}_log' for f in skewed_features]

scaler = StandardScaler()
df[features_to_scale] = scaler.fit_transform(df[features_to_scale])

In [6]:
# 6. Dimensionality Reduction
from sklearn.decomposition import PCA

# Reduce the 384-dimensional embeddings to capture the most variance with fewer features
n_components = 20
pca = PCA(n_components=n_components, random_state=42)

embedding_matrix = np.vstack(df['text_embedding'].values)
reduced_embeddings = pca.fit_transform(embedding_matrix)

print(f"Explained variance ratio by {n_components} components: {sum(pca.explained_variance_ratio_):.4f}")

# Add reduced embeddings back to dataframe
for i in range(n_components):
    df[f'emb_pca_{i}'] = reduced_embeddings[:, i]

# Drop the original full_text and complex text_embedding columns to keep dataset clean
df = df.drop(columns=['full_text', 'text_embedding'])

Explained variance ratio by 20 components: 0.6874


In [7]:
# 7. Feature Selection (lightweight)
from sklearn.ensemble import RandomForestClassifier

# Separate features and target
X = df.drop(columns=['outcome'])
y = df['outcome']

# Train a quick Random Forest to check feature importance
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X, y)

# Display top 10 most important features
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Top 10 features:\n", importances.head(10))

Top 10 features:
 sales_effectiveness        0.333715
customer_engagement        0.222958
engagement_per_turn        0.152061
engagement_x_length        0.086614
emb_pca_19                 0.007318
conversation_length_log    0.007075
emb_pca_11                 0.006961
emb_pca_4                  0.006582
conversation_length        0.006534
emb_pca_12                 0.006467
dtype: float64


In [8]:
# 8. Final Feature Set

# Keeping all engineered and reduced features for the final output
print(f"Final feature matrix shape: {df.shape}")
df.head()

Final feature matrix shape: (8000, 96)


,outcome,conversation_length,customer_engagement,sales_effectiveness,product_name_AR Renewable Insights,product_name_AutoLogistics Pro,product_name_BioSync,product_name_DataFlow Pro,product_name_DevOps Automation Suite,product_name_EcoSecure Manager,...,emb_pca_10,emb_pca_11,emb_pca_12,emb_pca_13,emb_pca_14,emb_pca_15,emb_pca_16,emb_pca_17,emb_pca_18,emb_pca_19
75721,1,12,0.492069,1.193471,False,False,False,False,False,False,...,0.066807,-0.122514,-0.107671,0.080654,-0.077675,0.099051,-0.125879,-0.080350,-0.102184,0.122516
80184,1,11,0.919787,1.510406,False,False,False,False,False,False,...,0.061330,0.118988,0.085169,0.019860,0.017449,0.086701,0.054027,-0.026589,-0.021728,0.073035
19864,1,10,0.492069,1.193471,False,False,False,False,False,False,...,0.074141,-0.085077,-0.093185,-0.078176,0.033623,0.037571,0.112593,0.054586,-0.033958,0.040120
76699,0,13,-0.363368,-0.708134,False,False,False,False,False,False,...,0.046244,-0.115017,-0.159607,0.069687,-0.058921,0.114048,-0.102885,-0.113755,-0.076834,0.103796
92991,1,9,0.492069,1.193471,False,True,False,False,False,False,...,-0.136776,-0.063576,0.052061,-0.120200,0.036002,-0.175387,-0.133748,0.068714,0.012956,0.224414


### Conclusion

- **Semantic Embeddings**: `all-MiniLM-L6-v2` captures the semantic nuance of conversations. By reducing the 384 dimensions via PCA, we retain core semantic variance while preventing the "curse of dimensionality" and reducing noise.
- **Interaction Features**: Logical combinations like `engagement_per_turn` or `text_per_turn` represent the density and quality of interaction, which intuitively matter more than just raw length.
- **Transformations**: Log-transforming skewed variables prevents extreme outliers from disproportionately influencing gradient-based or distance-based models. Standardization ensures continuous inputs operate on the same scale.
- **Simplicity & Explainability**: The pipeline avoids complex black-box aggregations. Every feature created naturally maps to a business-understandable concept, keeping the model cleanly explainable for a viva evaluation while achieving high predictive potential.